# POMDP Fire-Control Evaluation

POMDP v2 keeps the mixed `70/30` VAE pool as the source of plausible U-boat locations, but rebuilds firing parameters from noisy observations with `fire_control_lite`.

This tests whether a limited-information attacker can turn a plausible attack position into a usable firing solution without reusing the VAE candidate's original bearing/spread.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "convoy_sim").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from convoy_sim.feasibility import Environment
from convoy_sim.pomdp_fire_control import rebuild_records_with_fire_control, write_fire_control_candidate_pool
from convoy_sim.realism import get_attacker_observation_config
from experiments.evaluate_attack_candidate_pool import evaluate_candidate_pool, load_candidate_records
from experiments.run_pomdp_candidate_selector import run_belief_selector
from scenarios.convoy_profiles import get_convoy_layout_profile

METRIC_FIG_FACE = "lightgrey"
METRIC_AX_FACE = "white"
METRIC_GRID_COLOR = "lightgrey"


def style_metric_ax(ax, title: str, *, xlabel: str = "", ylabel: str = "", grid_axis: str = "y"):
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_facecolor(METRIC_AX_FACE)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, axis=grid_axis, color=METRIC_GRID_COLOR, alpha=0.45, linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)


## Configuration

Defaults match the POMDP v1 evaluation so v1/v2 comparisons are meaningful. `RUN_V1_EVAL` is off by default because the v1 notebook already produced matching runs.


In [ ]:
CANDIDATE_PATH = PROJECT_ROOT / "data" / "attack_profiles" / "vae_candidates" / "mixed_curated70_random30_hit_candidates.jsonl"
FULL_STATE_RUN_DIR = PROJECT_ROOT / "results" / "runs" / "candidate_pool_eval" / "20260512_144030_vae_final_baseline_mixed_vae"

OBSERVATION_PRESETS = ["good_contact", "baseline_night", "poor_contact"]
OBSERVATION_SEEDS = [1945, 1946, 1947, 1948, 1949]
MAX_PROFILES = 1000
TOP_K = 25

RUN_SELECTION = True
RUN_V1_EVAL = False
RUN_FIRE_CONTROL_REBUILD = True
RUN_FIRE_CONTROL_EVAL = True

CONVOY_PROFILE = "convoy_layout_1"
EVAL_SEEDS = [1942, 1943, 1944]
N_TRIALS_PER_SEED = 10
T_MAX = 400.0
OBJECTIVE_CFG = {"preset": "balanced_default"}

OUTPUT_ROOT = Path("results/runs")
DIAG_OUTPUT_DIR = PROJECT_ROOT / "results" / "diag" / "pomdp_fire_control_eval"
CANDIDATE_POOL_DIR = DIAG_OUTPUT_DIR / "candidate_pools"
DIAG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_POOL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
preset_rows = []
for preset in OBSERVATION_PRESETS:
    cfg = get_attacker_observation_config(preset)
    row = {"preset": preset}
    row.update(cfg.to_dict())
    preset_rows.append(row)

pd.DataFrame(preset_rows)


## Select VAE Candidate Locations

The belief selector is reused only to choose plausible locations/context under limited observation. POMDP v2 then rebuilds firing parameters from scratch.


In [ ]:
selection_runs: list[dict] = []

if RUN_SELECTION:
    for preset in OBSERVATION_PRESETS:
        for observation_seed in OBSERVATION_SEEDS:
            run_dir = run_belief_selector(
                candidate_path=CANDIDATE_PATH,
                project_root=PROJECT_ROOT,
                output_root=OUTPUT_ROOT,
                run_name=f"pomdp_v2_location_{preset}_seed{observation_seed}_notebook",
                convoy_profile=CONVOY_PROFILE,
                max_profiles=MAX_PROFILES,
                top_k=TOP_K,
                seed=int(observation_seed),
                observation_preset=preset,
            )
            selection_runs.append({
                "preset": preset,
                "observation_seed": int(observation_seed),
                "selection_dir": run_dir,
            })
            print(f"{preset} seed={observation_seed}: {run_dir}")
else:
    selection_runs = [
        # {"preset": "good_contact", "observation_seed": 1945, "selection_dir": PROJECT_ROOT / "results/runs/pomdp_candidate_selector/<existing_run>"},
    ]

pd.DataFrame(selection_runs)


## Rebuild Firing Solutions

For each selected location, the U-boat observes the convoy noisily and `fire_control_lite` rebuilds bearing, spread, torpedo speed, and run time. Source candidate hit/audit fields are not copied into the rebuilt records.


In [ ]:
ships = get_convoy_layout_profile(CONVOY_PROFILE).build_ships()
env = Environment(time_of_day="night", visibility_m=3500.0, sea_state=4)
rebuilt_runs: list[dict] = []

if RUN_FIRE_CONTROL_REBUILD:
    for item in selection_runs:
        preset = str(item["preset"])
        observation_seed = int(item["observation_seed"])
        selected_records = load_candidate_records(item["selection_dir"] / "top_belief_candidate_pool.jsonl")
        rebuilt_records = rebuild_records_with_fire_control(
            selected_records,
            ships=ships,
            seed=observation_seed,
            env=env,
            observation_preset=preset,
            profile_id_prefix=f"POMDP_FC_{preset.upper()}_{observation_seed}",
        )
        rebuilt_path = CANDIDATE_POOL_DIR / f"pomdp_fire_control_{preset}_seed{observation_seed}.jsonl"
        write_fire_control_candidate_pool(rebuilt_path, rebuilt_records)
        rebuilt_runs.append({
            "preset": preset,
            "observation_seed": observation_seed,
            "rebuilt_candidate_path": rebuilt_path,
            "profiles": len(rebuilt_records),
        })
        print(f"{preset} seed={observation_seed}: {rebuilt_path}")
else:
    rebuilt_runs = [
        # {"preset": "good_contact", "observation_seed": 1945, "rebuilt_candidate_path": PROJECT_ROOT / "results/diag/pomdp_fire_control_eval/candidate_pools/<existing_pool>.jsonl", "profiles": TOP_K},
    ]

pd.DataFrame(rebuilt_runs)


## Evaluate POMDP v2 Fire-Control Candidates

The rebuilt profiles are evaluated with the same Monte Carlo evaluator used by v1 and the full-state oracle.


In [ ]:
v2_eval_runs: list[dict] = []

if RUN_FIRE_CONTROL_EVAL:
    for item in rebuilt_runs:
        preset = str(item["preset"])
        observation_seed = int(item["observation_seed"])
        run_dir = evaluate_candidate_pool(
            candidate_path=item["rebuilt_candidate_path"],
            project_root=PROJECT_ROOT,
            output_root=OUTPUT_ROOT,
            run_name=f"pomdp_v2_fire_control_{preset}_seed{observation_seed}_top{TOP_K}_eval_notebook",
            convoy_profile=CONVOY_PROFILE,
            max_profiles=None,
            top_k=TOP_K,
            seeds=EVAL_SEEDS,
            n_trials_per_seed=N_TRIALS_PER_SEED,
            t_max=T_MAX,
            max_hits_per_torpedo=1,
            objective_cfg=OBJECTIVE_CFG,
        )
        v2_eval_runs.append({
            "selector": "pomdp_v2_fire_control",
            "preset": preset,
            "observation_seed": observation_seed,
            "eval_dir": run_dir,
        })
        print(f"{preset} seed={observation_seed}: {run_dir}")
else:
    v2_eval_runs = [
        # {"selector": "pomdp_v2_fire_control", "preset": "good_contact", "observation_seed": 1945, "eval_dir": PROJECT_ROOT / "results/runs/candidate_pool_eval/<existing_run>"},
    ]

pd.DataFrame(v2_eval_runs)


## Optional Matching POMDP v1 Evaluation

If matching v1 runs are not already available, set `RUN_V1_EVAL = True`. Otherwise this cell locates the latest existing v1 notebook runs for the same preset/seed grid.


In [ ]:
v1_eval_runs: list[dict] = []

if RUN_V1_EVAL:
    for item in selection_runs:
        preset = str(item["preset"])
        observation_seed = int(item["observation_seed"])
        run_dir = evaluate_candidate_pool(
            candidate_path=item["selection_dir"] / "top_belief_candidate_pool.jsonl",
            project_root=PROJECT_ROOT,
            output_root=OUTPUT_ROOT,
            run_name=f"pomdp_v1_selected_{preset}_seed{observation_seed}_top{TOP_K}_eval_notebook",
            convoy_profile=CONVOY_PROFILE,
            max_profiles=None,
            top_k=TOP_K,
            seeds=EVAL_SEEDS,
            n_trials_per_seed=N_TRIALS_PER_SEED,
            t_max=T_MAX,
            max_hits_per_torpedo=1,
            objective_cfg=OBJECTIVE_CFG,
        )
        v1_eval_runs.append({
            "selector": "pomdp_v1_selected_profile",
            "preset": preset,
            "observation_seed": observation_seed,
            "eval_dir": run_dir,
        })
else:
    for preset in OBSERVATION_PRESETS:
        for observation_seed in OBSERVATION_SEEDS:
            matches = sorted((PROJECT_ROOT / "results" / "runs" / "candidate_pool_eval").glob(
                f"*_pomdp_{preset}_seed{observation_seed}_top{TOP_K}_eval_notebook"
            ))
            if matches:
                v1_eval_runs.append({
                    "selector": "pomdp_v1_selected_profile",
                    "preset": preset,
                    "observation_seed": int(observation_seed),
                    "eval_dir": matches[-1],
                })

pd.DataFrame(v1_eval_runs)


## Summary Tables


In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

summary_rows = []
if FULL_STATE_RUN_DIR.exists():
    metrics = load_json(FULL_STATE_RUN_DIR / "metrics_summary.json")
    top = metrics["top_k"]
    best = metrics.get("best_candidate") or {}
    summary_rows.append({
        "selector": "full_state_oracle",
        "preset": "oracle",
        "observation_seed": -1,
        "profiles": int(top.get("profiles", TOP_K)),
        "expected_hits": float(top["expected_hits"]),
        "expected_loss": float(top.get("expected_loss", 0.0)),
        "expected_unique_ships_hit": float(top.get("expected_unique_ships_hit", 0.0)),
        "CVaR_90": float(top.get("CVaR_90", 0.0)),
        "CVaR_90_loss": float(top.get("CVaR_90_loss", 0.0)),
        "best_profile_id": str(best.get("profile_id", "")),
    })

for item in v1_eval_runs + v2_eval_runs:
    metrics = load_json(item["eval_dir"] / "metrics_summary.json")
    pool = metrics["candidate_pool"]
    best = metrics.get("best_candidate") or {}
    summary_rows.append({
        "selector": str(item["selector"]),
        "preset": str(item["preset"]),
        "observation_seed": int(item["observation_seed"]),
        "profiles": int(pool.get("profiles", TOP_K)),
        "expected_hits": float(pool["expected_hits"]),
        "expected_loss": float(pool.get("expected_loss", 0.0)),
        "expected_unique_ships_hit": float(pool.get("expected_unique_ships_hit", 0.0)),
        "CVaR_90": float(pool.get("CVaR_90", 0.0)),
        "CVaR_90_loss": float(pool.get("CVaR_90_loss", 0.0)),
        "best_profile_id": str(best.get("profile_id", "")),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
metric_cols = ["expected_hits", "expected_loss", "expected_unique_ships_hit", "CVaR_90", "CVaR_90_loss"]
run_df = summary_df.loc[summary_df["selector"] != "full_state_oracle"].copy()
agg_df = run_df.groupby(["selector", "preset"], as_index=False)[metric_cols].agg(["mean", "std", "min", "max"])
agg_df.columns = ["selector" if col[0] == "selector" else "preset" if col[0] == "preset" else f"{col[0]}_{col[1]}" for col in agg_df.columns]
agg_df = agg_df.reset_index(drop=True)

if FULL_STATE_RUN_DIR.exists():
    oracle_row = summary_df.loc[summary_df["selector"] == "full_state_oracle"].iloc[0]
    for metric in metric_cols:
        agg_df[f"oracle_gap_{metric}"] = float(oracle_row[metric]) - agg_df[f"{metric}_mean"]

agg_df


In [ ]:
if len(agg_df):
    plot_df = agg_df.copy()
    plot_df["label"] = plot_df["selector"].str.replace("pomdp_", "", regex=False) + "\n" + plot_df["preset"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), facecolor=METRIC_FIG_FACE)
    axes[0].bar(plot_df["label"], plot_df["expected_hits_mean"], yerr=plot_df["expected_hits_std"], color="#06768d", capsize=4)
    style_metric_ax(axes[0], "Expected Hits", ylabel="mean +/- std")
    axes[0].tick_params(axis="x", rotation=25)
    axes[1].bar(plot_df["label"], plot_df["expected_loss_mean"], yerr=plot_df["expected_loss_std"], color="#06768d", capsize=4)
    style_metric_ax(axes[1], "Expected Loss", ylabel="mean +/- std")
    axes[1].tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()


## Write Compact Artifacts


In [ ]:
summary_csv = DIAG_OUTPUT_DIR / "pomdp_fire_control_eval_summary.csv"
per_run_csv = DIAG_OUTPUT_DIR / "pomdp_fire_control_eval_per_run.csv"
rebuilt_runs_csv = DIAG_OUTPUT_DIR / "pomdp_fire_control_rebuilt_pools.csv"
summary_json = DIAG_OUTPUT_DIR / "pomdp_fire_control_eval_summary.json"

agg_df.to_csv(summary_csv, index=False)
summary_df.to_csv(per_run_csv, index=False)
pd.DataFrame(rebuilt_runs).to_csv(rebuilt_runs_csv, index=False)

summary_payload = {
    "workflow": "pomdp_fire_control_eval_notebook",
    "candidate_path": str(CANDIDATE_PATH.relative_to(PROJECT_ROOT)),
    "full_state_run_dir": str(FULL_STATE_RUN_DIR.relative_to(PROJECT_ROOT)),
    "top_k": int(TOP_K),
    "observation_presets": list(OBSERVATION_PRESETS),
    "observation_seeds": [int(seed) for seed in OBSERVATION_SEEDS],
    "metric_columns": list(metric_cols),
    "aggregate": json.loads(agg_df.to_json(orient="records")),
    "per_run": json.loads(summary_df.to_json(orient="records")),
    "rebuilt_runs": [
        {**item, "rebuilt_candidate_path": str(Path(item["rebuilt_candidate_path"]).relative_to(PROJECT_ROOT))}
        for item in rebuilt_runs
    ],
    "artifacts": {
        "summary_csv": summary_csv.name,
        "per_run_csv": per_run_csv.name,
        "rebuilt_runs_csv": rebuilt_runs_csv.name,
        "summary_json": summary_json.name,
    },
}
summary_json.write_text(json.dumps(summary_payload, indent=2) + "\n", encoding="utf-8")

{
    "summary_csv": str(summary_csv),
    "per_run_csv": str(per_run_csv),
    "rebuilt_runs_csv": str(rebuilt_runs_csv),
    "summary_json": str(summary_json),
}
